# Experimental forward-surrogate variant

This notebook records a later experimental modification developed while exploring the high-stiffness extension. It was **not** used to generate the soft-material results reported in the manuscript. The original soft-material training notebook was overwritten during later development; the matching checkpoint, training history, validation predictions, model architecture, and inference notebooks are provided in this repository.


### Forward Model Usage Guide - Conservative Improvement

This notebook demonstrates the process of training a forward model using the `ImprovedRegressionModel` class with **very conservative improvements**. The model is trained on pre-split augmented data to ensure consistency with the diffusion model training pipeline.

#### **CONSERVATIVE IMPROVEMENTS:**
- **Model Architecture**: Hidden dimension 256 (unchanged), head/distribution dropout 0.15 and trunk dropout 0.20
- **Training Parameters**: Learning rate 1e-4 (unchanged), weight decay 4e-5
- **Training Strategy**: 100 epochs, cosine annealing scheduler (unchanged)
- **No Early Stopping**: Keep original training behavior
- **Goal**: Minimal changes to reduce validation gap while maintaining high performance

#### Key Steps:
1. **Data Loading**: The dataset is loaded from pre-split files:
   - `combined_augmented_train.xlsx`: Training set (already split in RNN augmentation)
   - `val.xlsx`: Validation set (already split in RNN augmentation)
   - This ensures consistency with the diffusion model training and prevents data leakage
2. **Scaler Loading**: Pre-fitted scalers (`scaler_x_new.pkl` and `scaler_y_new.pkl`) are loaded to ensure:
   - Consistent standardization across all models (RNN, diffusion, forward)
   - Scalers fitted only on training data (no data leakage)
3. **Model Training**: The improved `ImprovedRegressionModel` is trained using the pre-split training dataset with enhanced regularization without early stopping.
4. **Evaluation**: The model's performance is evaluated on both training and validation datasets with improved generalization.

#### Data Pipeline Consistency:
- **RNN Augmentation** → Augment data, split into train/val, fit scalers
- **Diffusion Model** → Load pre-split data and scalers 
- **Forward Model** → Load pre-split data and scalers 

#### Module Imports:
- **pandas**: Used for data manipulation and loading the dataset.
- **torch**: Provides the framework for building and training the neural network model.
- **sklearn**: Used for data preprocessing and evaluation metrics.
- **matplotlib**: Utilized for visualizing the training and validation loss curves.


### Environment Settings

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import joblib
from matplotlib.ticker import MultipleLocator
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [ ]:
# Use serif + bold global settings before drawing
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif']  = ['DejaVu Serif', 'Liberation Serif']
plt.rcParams['font.weight'] = 'bold'

### 1. Define the forward model


In [ ]:
class ImprovedRegressionModel(nn.Module):
    def __init__(self,
                 moduli_dim: int = 6,
                 dist_dim: int = 2,
                 hidden_dim: int = 256,
                 trunk_layers: int = 2,
                 dropout: float = 0.15,
                 trunk_dropout: float = 0.20):  # Stronger regularization on the conditional (trunk) branch
        """
        The input x is composed of two parts:
          - x_mod (batch, 6): Fixed 6-column material moduli
          - x_dist (batch, 2): Distribution values predicted by the diffusion model
        The outputs y0, y1 are two effective moduli.
        """
        super().__init__()
        assert dist_dim == 2, "This model expects dist_dim=2"
        self.moduli_dim = moduli_dim
        self.dist_dim   = dist_dim

        # --- 1) Trunk for processing 6-dimensional moduli ---
        trunk = []
        in_dim = moduli_dim
        for _ in range(trunk_layers):
            trunk.append(nn.Linear(in_dim, hidden_dim))
            trunk.append(nn.LayerNorm(hidden_dim))
            trunk.append(nn.GELU())
            trunk.append(nn.Dropout(trunk_dropout))  # Stronger regularization on conditional features
            in_dim = hidden_dim
        self.trunk = nn.Sequential(*trunk)  # output (batch, hidden_dim)

        # --- 2) Simple mapping for processing distribution x_dist ---
        #    Directly map (batch,2) to (batch, hidden_dim//4), then concatenate to trunk output
        self.dist_embed = nn.Sequential(
            nn.Linear(dist_dim, hidden_dim // 4),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        # --- 3) Fusion dimension after concatenating trunk and dist_embed outputs ---
        fusion_dim = hidden_dim + (hidden_dim // 4)
        self.fusion_norm = nn.LayerNorm(fusion_dim)

        # --- 4) Head for storage modulus (y0) ---
        self.head0 = nn.Sequential(
            nn.Linear(fusion_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

        # --- 5) Head for loss modulus (y1): deeper network ---
        self.head1 = nn.Sequential(
            nn.Linear(fusion_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 4, 1)
        )

        # --- 6) Learn variance parameters (optional for NLL loss) ---
        self.log_sigma0 = nn.Parameter(torch.zeros(()))
        self.log_sigma1 = nn.Parameter(torch.zeros(()))

        # --- 7) Initialize all linear layer weights ---
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (batch, 8) = [ x_mod (6 dims) | x_dist (2 dims) ]
        Returns (batch, 2): first column y0, second column y1
        """
        # Split
        x_mod  = x[:, : self.moduli_dim]  # (batch, 6)
        x_dist = x[:, self.moduli_dim:]   # (batch, 2)

        # 1) First, send moduli into the trunk
        z_mod = self.trunk(x_mod)         # (batch, hidden_dim)

        # 2) Send distribution into dist_embed
        z_dist = self.dist_embed(x_dist)  # (batch, hidden_dim//4)

        # 3) Concatenate features + LayerNorm
        z = torch.cat([z_mod, z_dist], dim=1)  # (batch, hidden_dim + hidden_dim//4)
        z = self.fusion_norm(z)

        # 4) Pass through two heads to get y0, y1
        y0 = self.head0(z)  # (batch, 1)
        y1 = self.head1(z)  # (batch, 1)

        return torch.cat([y0, y1], dim=-1)  # (batch, 2)


### 2. Data preparation: Load pre-split data and scalers

In [ ]:
# 1) Load pre-split data (consistent with diffusion model training)
train_path = "data/combined_augmented_train.xlsx"
val_path = "data/val.xlsx"

print("Loading pre-split data...")
df_train = pd.read_excel(train_path, header=0)
df_val = pd.read_excel(val_path, header=0)

X_tr = df_train.iloc[:, :8].values   # (N_train, 8)
Y_tr = df_train.iloc[:, 8:].values   # (N_train, 2)
X_val = df_val.iloc[:, :8].values    # (N_val, 8)
Y_val = df_val.iloc[:, 8:].values    # (N_val, 2)

print(f"Train set: {X_tr.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")

# 2) Load pre-fitted scalers (fitted on training set during RNN augmentation)
scaler_x_path = "artifacts/experimental/scaler_x_new.pkl"
scaler_y_path = "artifacts/experimental/scaler_y_new.pkl"

print(f"\nLoading pre-fitted scalers...")
scaler_x = joblib.load(scaler_x_path)
scaler_y = joblib.load(scaler_y_path)

print(f"Loaded scaler_x from: {scaler_x_path}")
print(f"Loaded scaler_y from: {scaler_y_path}")

# Verify scaler parameters
print(f"\nScaler parameters (first 3 features):")
print(f"  X mean: {scaler_x.mean_[:3]}")
print(f"  X std:  {scaler_x.scale_[:3]}")
print(f"  Y mean: {scaler_y.mean_}")
print(f"  Y std:  {scaler_y.scale_}")

# 3) Transform both splits using pre-fitted scalers
X_tr_scaled = scaler_x.transform(X_tr)
Y_tr_scaled = scaler_y.transform(Y_tr)
X_val_scaled = scaler_x.transform(X_val)
Y_val_scaled = scaler_y.transform(Y_val)

# 4) Convert to tensors & DataLoaders
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")

train_ds = TensorDataset(
    torch.tensor(X_tr_scaled, dtype=torch.float32, device=device),
    torch.tensor(Y_tr_scaled, dtype=torch.float32, device=device)
)
val_ds = TensorDataset(
    torch.tensor(X_val_scaled, dtype=torch.float32, device=device),
    torch.tensor(Y_val_scaled, dtype=torch.float32, device=device)
)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=128, shuffle=False)

print(f"\nData preparation completed!")
print(f"✓ Using pre-split data (consistent with diffusion model)")
print(f"✓ Using pre-fitted scalers (no data leakage)")

In [ ]:
def eval_loader_losses(model, loader, criterion, device):
    """
    Returns the MSE for each head on the loader (with dropout turned off).
    """
    model.eval()  # Ensure Dropout/LayerNorm is in eval mode
    total0 = total1 = count = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            pred   = model(xb)           # (batch, 2)
            l0 = criterion(pred[:, 0], yb[:, 0])
            l1 = criterion(pred[:, 1], yb[:, 1])
            b  = xb.size(0)
            total0 += l0.item() * b
            total1 += l1.item() * b
            count  += b
    return total0 / count, total1 / count

def eval_loader_loss(model, loader, criterion, device):
    """
    Returns the overall MSE on the loader (combined MSE of two heads, with dropout turned off).
    """
    model.eval()
    total = count = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            pred   = model(xb)  # (batch, 2)
            total += criterion(pred, yb).item() * xb.size(0)
            count += xb.size(0)
    return total / count

### 3. Forward model training

In [ ]:
# Hyperparameters, model, optimizer, and scheduler - CONSERVATIVE IMPROVEMENT
epochs = 100  # Keep original number of training epochs
temp   = 0.5
dwa_warmup_epochs = 5
storage_bias = 1.2
loss_bias = 0.8
cond_noise_std = 0.02  # Noise augmentation for conditional inputs
trunk_dropout = 0.20

# Conservative improved model configuration
forward_model = ImprovedRegressionModel(
    moduli_dim=6,
    dist_dim=2,
    hidden_dim=256,
    trunk_layers=2,
    dropout=0.15,
    trunk_dropout=trunk_dropout
)
forward_model.to(device)

# Conservative improved optimizer configuration
optimizer = optim.AdamW(forward_model.parameters(), lr=1e-4, weight_decay=4e-5)  # Further increase weight decay
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)
criterion = nn.MSELoss()

# Conservative improved configuration
print(f"Conservative Improved Training Configuration:")
print(f"  Epochs: {epochs}")
print(f"  Learning rate: 1e-4 (unchanged)")
print(f"  Weight decay: 4e-5 (moderate increase from 1e-5)")
print(f"  Dropout(head/dist): 0.15")
print(f"  Trunk dropout: {trunk_dropout}")
print(f"  Cond input noise std: {cond_noise_std}")
print(f"  DWA warmup epochs: {dwa_warmup_epochs}")
print(f"  DWA bias (storage, loss): ({storage_bias}, {loss_bias})")
print(f"  Hidden dim: 256 (unchanged)")
print(f"  DWA temperature: {temp}")
print(f"  No early stopping (keep original behavior)")

# DWA & record curves
train_loss0, train_loss1     = [], []
val_loss0,   val_loss1       = [], []

# Record MSE for each head in 'eval mode'
train_eval0, train_eval1     = [], []
val_eval0,   val_eval1       = [], []

# (If you want to keep the original overall eval curve)
train_eval_losses, val_eval_losses = [], []

# === Data structures for saving training history ===
training_history = {
    'epoch': [],
    'train_loss0': [],
    'train_loss1': [],
    'val_loss0': [],
    'val_loss1': [],
    'train_eval0': [],
    'train_eval1': [],
    'val_eval0': [],
    'val_eval1': [],
    'train_eval_combined': [],
    'val_eval_combined': [],
    'learning_rate': [],
    'dwa_weight0': [],
    'dwa_weight1': []
}

for epoch in range(epochs):
    # —— 1) Calculate DWA weights w0, w1 ——
    if epoch < dwa_warmup_epochs:
        raw_w0 = storage_bias
        raw_w1 = loss_bias
    elif len(train_loss0) >= 2 and len(train_loss1) >= 2:
        r0 = train_loss0[-1] / max(train_loss0[-2], 1e-8)
        r1 = train_loss1[-1] / max(train_loss1[-2], 1e-8)
        raw_w0 = np.exp(r0 / temp)
        raw_w1 = np.exp(r1 / temp)
    else:
        raw_w0 = raw_w1 = 1.0
    s = raw_w0 + raw_w1
    w0 = raw_w0 / s
    w1 = raw_w1 / s

    # —— 2) Training phase ——
    forward_model.train()  # Dropout 
    sum0 = sum1 = count = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        if cond_noise_std > 0:
            noise = torch.randn_like(xb[:, :forward_model.moduli_dim]) * cond_noise_std
            xb_aug = xb.clone()
            xb_aug[:, :forward_model.moduli_dim] += noise
        else:
            xb_aug = xb

        pred = forward_model(xb_aug)  # (batch, 2)

        loss0 = criterion(pred[:, 0], yb[:, 0])
        loss1 = criterion(pred[:, 1], yb[:, 1])

        # Use DWA weights instead of learned sigma
        loss = w0 * loss0 + w1 * loss1

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(forward_model.parameters(), max_norm=1.0)
        optimizer.step()

        b = xb.size(0)
        sum0  += loss0.item() * b
        sum1  += loss1.item() * b
        count += b

    train0 = sum0 / count
    train1 = sum1 / count
    train_loss0.append(train0)
    train_loss1.append(train1)

    # —— 3) Validation phase ——
    forward_model.eval()  # trun off Dropout
    v0 = v1 = count = 0

    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = forward_model(xb)
            l0   = criterion(pred[:, 0], yb[:, 0]).item()
            l1   = criterion(pred[:, 1], yb[:, 1]).item()
            b    = xb.size(0)
            v0  += l0 * b
            v1  += l1 * b
            count += b

    val0 = v0 / count
    val1 = v1 / count
    val_loss0.append(val0)
    val_loss1.append(val1)

    # —— 4) eval_loader_losses records MSE for each head on train/val ——
    train_l0_eval, train_l1_eval = eval_loader_losses(
        forward_model, train_loader, criterion, device
    )
    val_l0_eval,   val_l1_eval   = eval_loader_losses(
        forward_model, val_loader,   criterion, device
    )
    train_eval0.append(train_l0_eval)
    train_eval1.append(train_l1_eval)
    val_eval0  .append(val_l0_eval)
    val_eval1  .append(val_l1_eval)

    # —— 5) eval_loader_loss records overall train/val MSE ——
    overall_train_eval = eval_loader_loss(
        forward_model, train_loader, criterion, device
    )
    overall_val_eval   = eval_loader_loss(
        forward_model, val_loader,   criterion, device
    )
    train_eval_losses.append(overall_train_eval)
    val_eval_losses  .append(overall_val_eval)

    # —— 6) Print metrics ——
    ratio = (epoch + 1) / epochs
    print(
        f"Epoch {epoch+1}/{epochs} ({ratio:.2%})  "
        f"w0={w0:.3f}, w1={w1:.3f}  |  "
        f"Train(storage)={train0:.4e}, Train(loss)={train1:.4e}  |  "
        f"Val(storage)={val_loss0[-1]:.4e}, Val(loss)={val_loss1[-1]:.4e}  |  "
        f"TrainEval(storage)={train_l0_eval:.4e}, TrainEval(loss)={train_l1_eval:.4e}  |  "
        f"ValEval(storage)={val_l0_eval:.4e}, ValEval(loss)={val_l1_eval:.4e}  |  "
        f"OverallTrainEval={overall_train_eval:.4e}, OverallValEval={overall_val_eval:.4e}"
    )

    # —— 7) Learning rate scheduling ——
    scheduler.step()
    
    # —— 8) Record training history ——
    lr_current = optimizer.param_groups[0]['lr']
    training_history['epoch'].append(epoch + 1)
    training_history['train_loss0'].append(train0)
    training_history['train_loss1'].append(train1)
    training_history['val_loss0'].append(val0)
    training_history['val_loss1'].append(val1)
    training_history['train_eval0'].append(train_l0_eval)
    training_history['train_eval1'].append(train_l1_eval)
    training_history['val_eval0'].append(val_l0_eval)
    training_history['val_eval1'].append(val_l1_eval)
    training_history['train_eval_combined'].append(overall_train_eval)
    training_history['val_eval_combined'].append(overall_val_eval)
    training_history['learning_rate'].append(lr_current)
    training_history['dwa_weight0'].append(w0)
    training_history['dwa_weight1'].append(w1)

# save model - CONSERVATIVE IMPROVEMENT
base_dir = Path("artifacts")
base_dir.mkdir(parents=True, exist_ok=True)

torch.save(forward_model.state_dict(), base_dir/"forward_model_weights_conservative.pt")
print(f"\nConservative improved model saved to: {base_dir/'forward_model_weights_conservative.pt'}")

# === Save training history to CSV file ===
artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)

history_df = pd.DataFrame(training_history)
history_df.to_csv(artifacts_dir/'forward_model_training_history_conservative.csv', index=False)
print(f"Conservative training history saved to: {artifacts_dir/'forward_model_training_history_conservative.csv'}")

### 4. Generate predictions and save data for scatter plots

In [ ]:
# Use validation set for model inference
# Note: X_val_scaled and Y_val_scaled are already transformed using pre-fitted scalers

# Convert scaled validation data to tensors
x_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32, device=device)
y_val_tensor = torch.tensor(Y_val_scaled, dtype=torch.float32, device=device)

# Model inference
forward_model.eval()
with torch.no_grad():
    preds_std = forward_model(x_val_tensor)  # Predictions in standardized space
preds_std = preds_std.cpu().numpy()

# Inverse transform back to original physical units
preds_orig = scaler_y.inverse_transform(preds_std)  # (N,2)
y_true_orig = scaler_y.inverse_transform(y_val_tensor.cpu().numpy())  # (N,2)

# Save prediction data for scatter plots
import pickle
prediction_data = {
    'y_true': y_true_orig,
    'y_pred': preds_orig
}

with open(artifacts_dir/'forward_model_predictions_conservative.pkl', 'wb') as f:
    pickle.dump(prediction_data, f)

print(f"Conservative prediction data saved to: {artifacts_dir/'forward_model_predictions_conservative.pkl'}")
print(f"  - Validation samples: {len(y_true_orig)}")

### 5. Model Evaluation Function

This section defines the `evaluate_model` function, which is used to assess the performance of the trained model on both the training and validation datasets. The function calculates the mean squared error (MSE) and the coefficient of determination (R²) for each dataset, providing insights into the model's accuracy and generalization ability.


In [ ]:
def evaluate_model(forward_model, train_loader, val_loader, criterion, device):
    """
    Evaluate the model on both training and validation datasets.
    
    NOTE: R² must be calculated on the entire dataset, not as a batch average!
    This is because R² is a non-linear metric based on sum of squared residuals.
    """
    forward_model.eval()  # Set the model to evaluation mode

    # Collect all predictions and true values
    all_train_preds = []
    all_train_true = []
    all_val_preds = []
    all_val_true = []
    
    train_loss = 0.0
    val_loss = 0.0
    train_count = 0
    val_count = 0

    # Evaluate on training data
    with torch.no_grad():
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = forward_model(xb)
            loss = criterion(pred, yb)
            train_loss += loss.item() * xb.size(0)
            train_count += xb.size(0)
            
            # Collect predictions and true values for R² calculation
            all_train_preds.append(pred.cpu().numpy())
            all_train_true.append(yb.cpu().numpy())

    # Evaluate on validation data
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = forward_model(xb)
            loss = criterion(pred, yb)
            val_loss += loss.item() * xb.size(0)
            val_count += xb.size(0)
            
            # Collect predictions and true values for R² calculation
            all_val_preds.append(pred.cpu().numpy())
            all_val_true.append(yb.cpu().numpy())

    # Calculate average loss
    train_loss /= train_count
    val_loss /= val_count
    
    # Concatenate all batches
    all_train_preds = np.vstack(all_train_preds)
    all_train_true = np.vstack(all_train_true)
    all_val_preds = np.vstack(all_val_preds)
    all_val_true = np.vstack(all_val_true)
    
    # Calculate R² on the entire dataset (correct method)
    train_r2 = r2_score(all_train_true, all_train_preds)
    val_r2 = r2_score(all_val_true, all_val_preds)
    
    # Calculate R² for each output dimension
    train_r2_dim0 = r2_score(all_train_true[:, 0], all_train_preds[:, 0])
    train_r2_dim1 = r2_score(all_train_true[:, 1], all_train_preds[:, 1])
    val_r2_dim0 = r2_score(all_val_true[:, 0], all_val_preds[:, 0])
    val_r2_dim1 = r2_score(all_val_true[:, 1], all_val_preds[:, 1])

    print("="*70)
    print("Model Evaluation Results")
    print("="*70)
    print(f"Training Loss:   {train_loss:.4f}")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"\nOverall R²:")
    print(f"  Training R²:   {train_r2:.4f}")
    print(f"  Validation R²: {val_r2:.4f}")
    print(f"\nR² by dimension:")
    print(f"  Training   - E'_H:  {train_r2_dim0:.4f}, E''_H: {train_r2_dim1:.4f}")
    print(f"  Validation - E'_H:  {val_r2_dim0:.4f}, E''_H: {val_r2_dim1:.4f}")
    print("="*70)
    
    return train_loss, val_loss, train_r2, val_r2

# Example usage
train_loss, val_loss, train_r2, val_r2 = evaluate_model(
    forward_model, train_loader, val_loader, nn.MSELoss(), device
)


### 6. Reload training data and generate plots

**Important**: This section should be run after training is complete to visualize the results.

The training process saves all necessary data (training history and predictions) to files. To generate plots, simply run the cells below. This allows you to:
- Regenerate plots without retraining the model
- Customize plot parameters without affecting the saved data
- Generate plots multiple times as needed

In [ ]:
# Load conservative training history from saved CSV file
import pandas as pd
import pickle
from pathlib import Path

artifacts_dir = Path("artifacts")
history_file = artifacts_dir / 'forward_model_training_history_conservative.csv'
prediction_file = artifacts_dir / 'forward_model_predictions_conservative.pkl'

# Check if files exist
if not history_file.exists() or not prediction_file.exists():
    print("=" * 70)
    print("ERROR: Required conservative data files not found!")
    print("=" * 70)
    if not history_file.exists():
        print(f"Missing: {history_file}")
    if not prediction_file.exists():
        print(f"Missing: {prediction_file}")
    print("\nThese files are generated during conservative training.")
    print("Please run Cells 1-12 first to train the conservative model and save data.")
    print("=" * 70)
    raise FileNotFoundError("Run conservative training cells first (Cells 1-12)")

# Load training history
history_df = pd.read_csv(history_file)

# Extract data for plotting
train_eval0 = history_df['train_eval0'].tolist()
train_eval1 = history_df['train_eval1'].tolist()
val_eval0 = history_df['val_eval0'].tolist()
val_eval1 = history_df['val_eval1'].tolist()
train_eval_losses = history_df['train_eval_combined'].tolist()
val_eval_losses = history_df['val_eval_combined'].tolist()

print(f"✓ Loaded conservative training history: {len(history_df)} epochs")
print(f"✓ Conservative training completed from epoch 1 to {history_df['epoch'].max()}")

In [ ]:
# Regenerate loss curves from loaded data

import matplotlib.pyplot as plt

# Use serif + bold global settings
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif']  = ['DejaVu Serif', 'Liberation Serif']
plt.rcParams['font.weight'] = 'bold'

epochs = list(range(1, len(train_eval0) + 1))

# Storage Modulus
plt.figure(figsize=(8,8))
plt.plot(epochs, train_eval0, label="Train", linewidth=3)
plt.plot(epochs, val_eval0, label="Validation", linewidth=3)
plt.xlabel("Epoch", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=10)
plt.ylabel("Loss ($E_H^{\\prime}$)",   fontsize=26, fontweight='bold', fontfamily='serif', labelpad=6)
plt.xticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.yticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.legend(prop={'size': 24, 'weight': 'bold', 'family': 'serif'})
plt.grid(False)

ax = plt.gca()
for spine in ax.spines.values():
    spine.set_linewidth(3)
plt.tick_params(axis='both', which='major', length=8, width=3)

plt.tight_layout()
plt.show()

# Loss Modulus
plt.figure(figsize=(8,8))
plt.plot(epochs, train_eval1, label="Train", linewidth=3)
plt.plot(epochs, val_eval1, label="Validation", linewidth=3)
plt.xlabel("Epoch", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=10)
plt.ylabel("Loss ($E_H^{\\prime\\prime}$)",   fontsize=26, fontweight='bold', fontfamily='serif', labelpad=6)
plt.xticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.yticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.legend(prop={'size': 24, 'weight': 'bold', 'family': 'serif'})
plt.grid(False)

ax = plt.gca()
for spine in ax.spines.values():
    spine.set_linewidth(3)
plt.tick_params(axis='both', which='major', length=8, width=3)

plt.tight_layout()
plt.show()

# Overall Model
plt.figure(figsize=(8,8))
plt.plot(epochs, train_eval_losses, label="Train", linewidth=3)
plt.plot(epochs, val_eval_losses, label="Validation", linewidth=3)
plt.xlabel("Epoch", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=10)
plt.ylabel("MSE Loss",   fontsize=28, fontweight='bold', fontfamily='serif', labelpad=6)
plt.xticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.yticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.legend(prop={'size': 24, 'weight': 'bold', 'family': 'serif'})
plt.grid(False)

ax = plt.gca()
#ax.yaxis.set_major_locator(MultipleLocator(base=0.05))
for spine in ax.spines.values():
    spine.set_linewidth(3)
plt.tick_params(axis='both', which='major', length=8, width=3)

plt.show()

print("Loss curves generated from saved training history")

In [ ]:
# Load conservative prediction data and regenerate scatter plots
import numpy as np
from sklearn.metrics import r2_score

# Load saved conservative prediction data
with open(artifacts_dir/'forward_model_predictions_conservative.pkl', 'rb') as f:
    prediction_data = pickle.load(f)

y_true_orig = prediction_data['y_true']
preds_orig = prediction_data['y_pred']

print(f"Loaded conservative prediction data: {len(y_true_orig)} samples")

# Regenerate scatter plots
def parity_plot_with_r2_annotation(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    names=("$E_H^{\\prime}$ (MPa)", "$E_H^{\\prime\\prime}$ (MPa)"),
    point_size=2
):
    colors = ['tab:blue', 'tab:orange']  # Colors for points & lines in different plots
    colors_line = ['tab:red', 'tab:blue']  

    for i in range(2):
        yt = y_true[:, i]
        yp = y_pred[:, i]
        r2 = r2_score(yt, yp)

        plt.figure(figsize=(12, 8))
        ax = plt.gca()

        # scatter plot with color
        ax.scatter(yt, yp,c=colors[i],edgecolor='none',alpha=0.7,s=point_size,label=names[i])

        # parity line in same color
        mn, mx = min(yt.min(), yp.min()), max(yt.max(), yp.max())
        ax.plot([mn, mx], [mn, mx],'--',color=colors_line[i],linewidth=4)

        # add R² annotation
        ax.text(0.05, 0.95, f"R² = {r2:.3f}",
            transform=plt.gca().transAxes,
            fontsize=28, fontweight='bold', family='serif',
            verticalalignment='top')

        # labels
        ax.set_xlabel(f"True {names[i]}", fontsize=28, fontweight='bold', family='serif', labelpad=10)
        ax.set_ylabel(f"Pred {names[i]}", fontsize=28, fontweight='bold', family='serif', labelpad=12)

        # tick styling
        ax.tick_params(axis='both', which='major', labelsize=24)
        for tick in ax.get_xticklabels() + ax.get_yticklabels():
            tick.set_fontweight('bold')
            tick.set_fontfamily('serif')

        ax = plt.gca()
        for spine in ax.spines.values():
            spine.set_linewidth(3)
        plt.tick_params(axis='both', which='major', length=8, width=3)

        plt.tight_layout()
        plt.show()

parity_plot_with_r2_annotation(y_true_orig, preds_orig, point_size=10)
print("Conservative scatter plots generated from saved prediction data")
